In [1]:
!ls data/IoTID20 

 dataset_description.xlsx  'IoT Network Intrusion Dataset.csv'


# Discovery


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("data/IoTID20/IoT Network Intrusion Dataset.csv")
assert DATA_PATH.exists(), f"Cannot find dataset at: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

print("Shape:", df.shape)
print("Number of columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
display(df.dtypes.to_frame("dtype").T)

print("\nHead:")
display(df.head())

print("\nMissing values per column:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique values per column:")
display(df.nunique().sort_values().to_frame("nunique"))

label_cols = [c for c in ["Label", "Cat", "Sub_Cat"] if c in df.columns]
for col in label_cols:
    print(f"\nValue counts for {col}:")
    counts = df[col].value_counts(dropna=False)
    ratios = (counts / len(df)).round(4)
    display(pd.DataFrame({"count": counts, "ratio": ratios}))

special_cols = [c for c in ["Flow_ID", "Src_IP", "Dst_IP", "Timestamp", "Src_Port", "Dst_Port"] if c in df.columns]
print("\nSpecial columns to inspect:")
print(special_cols)

if special_cols:
    display(df[special_cols].head())

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("\nCheck for inf / -inf in numeric columns:")
inf_counts = pd.Series(
    {col: np.isinf(df[col]).sum() for col in numeric_cols}
).sort_values(ascending=False)
display(inf_counts[inf_counts > 0].to_frame("inf_count"))

print("\nNumeric summary:")
display(df[numeric_cols].describe().T)

print("\nSelected quantiles for numeric columns:")
quantiles = df[numeric_cols].quantile([0.00, 0.01, 0.25, 0.50, 0.75, 0.99, 1.00]).T
display(quantiles)

print("\nConstant columns:")
constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
print(constant_cols)

print("\nNear-constant columns (top value > 99%):")
near_constant = {}
for col in df.columns:
    top_ratio = df[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_ratio > 0.99:
        near_constant[col] = top_ratio
display(pd.DataFrame.from_dict(near_constant, orient="index", columns=["top_value_ratio"]).sort_values("top_value_ratio", ascending=False))

print("\nZero ratio in numeric columns:")
zero_ratio = pd.Series(
    {col: (df[col] == 0).mean() for col in numeric_cols}
).sort_values(ascending=False)
display(zero_ratio.to_frame("zero_ratio"))


Shape: (625783, 86)
Number of columns: 86

Columns:
['Flow_ID', 'Src_IP', 'Src_Port', 'Dst_IP', 'Dst_Port', 'Protocol', 'Timestamp', 'Flow_Duration', 'Tot_Fwd_Pkts', 'Tot_Bwd_Pkts', 'TotLen_Fwd_Pkts', 'TotLen_Bwd_Pkts', 'Fwd_Pkt_Len_Max', 'Fwd_Pkt_Len_Min', 'Fwd_Pkt_Len_Mean', 'Fwd_Pkt_Len_Std', 'Bwd_Pkt_Len_Max', 'Bwd_Pkt_Len_Min', 'Bwd_Pkt_Len_Mean', 'Bwd_Pkt_Len_Std', 'Flow_Byts/s', 'Flow_Pkts/s', 'Flow_IAT_Mean', 'Flow_IAT_Std', 'Flow_IAT_Max', 'Flow_IAT_Min', 'Fwd_IAT_Tot', 'Fwd_IAT_Mean', 'Fwd_IAT_Std', 'Fwd_IAT_Max', 'Fwd_IAT_Min', 'Bwd_IAT_Tot', 'Bwd_IAT_Mean', 'Bwd_IAT_Std', 'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_PSH_Flags', 'Bwd_PSH_Flags', 'Fwd_URG_Flags', 'Bwd_URG_Flags', 'Fwd_Header_Len', 'Bwd_Header_Len', 'Fwd_Pkts/s', 'Bwd_Pkts/s', 'Pkt_Len_Min', 'Pkt_Len_Max', 'Pkt_Len_Mean', 'Pkt_Len_Std', 'Pkt_Len_Var', 'FIN_Flag_Cnt', 'SYN_Flag_Cnt', 'RST_Flag_Cnt', 'PSH_Flag_Cnt', 'ACK_Flag_Cnt', 'URG_Flag_Cnt', 'CWE_Flag_Count', 'ECE_Flag_Cnt', 'Down/Up_Ratio', 'Pkt_Size_Avg', 'Fwd_Seg

,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Timestamp,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,...,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Cat,Sub_Cat
dtype,object,object,int64,object,int64,int64,object,int64,int64,int64,...,float64,float64,float64,float64,float64,float64,float64,object,object,object



Head:


,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Timestamp,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,...,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Cat,Sub_Cat
0,192.168.0.13-192.168.0.16-10000-10101-17,192.168.0.13,10000,192.168.0.16,10101,17,25/07/2019 03:25:53 AM,75,1,1,...,0.0,0.0,0.0,75.0,0.000000,75.0,75.0,Anomaly,Mirai,Mirai-Ackflooding
1,192.168.0.13-222.160.179.132-554-2179-6,222.160.179.132,2179,192.168.0.13,554,6,26/05/2019 10:11:06 PM,5310,1,2,...,0.0,0.0,0.0,2655.0,2261.327486,4254.0,1056.0,Anomaly,DoS,DoS-Synflooding
2,192.168.0.13-192.168.0.16-9020-52727-6,192.168.0.16,52727,192.168.0.13,9020,6,11/07/2019 01:24:48 AM,141,0,3,...,0.0,0.0,0.0,70.5,0.707107,71.0,70.0,Anomaly,Scan,Scan Port OS
3,192.168.0.13-192.168.0.16-9020-52964-6,192.168.0.16,52964,192.168.0.13,9020,6,04/09/2019 03:58:17 AM,151,0,2,...,0.0,0.0,0.0,151.0,0.000000,151.0,151.0,Anomaly,Mirai,Mirai-Hostbruteforceg
4,192.168.0.1-239.255.255.250-36763-1900-17,192.168.0.1,36763,239.255.255.250,1900,17,10/09/2019 01:41:18 AM,153,2,1,...,0.0,0.0,0.0,76.5,0.707107,77.0,76.0,Anomaly,Mirai,Mirai-Hostbruteforceg



Missing values per column:


...

,0.00,0.01,0.25,0.50,0.75,0.99,1.00
Src_Port,0.0,0.0,9020.0,51991.0,56361.000000,64781.000000,65500.000000
Dst_Port,0.0,0.0,8899.0,9020.0,10101.000000,56196.000000,65371.000000
Protocol,0.0,0.0,6.0,6.0,17.000000,17.000000,17.000000
Flow_Duration,0.0,3.0,76.0,132.0,221.000000,8344.180000,99984.000000
Tot_Fwd_Pkts,0.0,0.0,0.0,1.0,2.000000,21.000000,186.000000
...,...,...,...,...,...,...,...
Active_Min,0.0,0.0,0.0,0.0,0.000000,2.000000,6659.000000
Idle_Mean,0.0,0.0,73.0,93.5,141.000000,7804.180000,99973.000000
Idle_Std,0.0,0.0,0.0,0.0,1.527525,173.241161,67071.906623
Idle_Max,0.0,0.0,74.0,114.0,154.000000,8038.180000,99973.000000



Constant columns:
['Fwd_PSH_Flags', 'Fwd_URG_Flags', 'Fwd_Byts/b_Avg', 'Fwd_Pkts/b_Avg', 'Fwd_Blk_Rate_Avg', 'Bwd_Byts/b_Avg', 'Bwd_Pkts/b_Avg', 'Bwd_Blk_Rate_Avg', 'Init_Fwd_Win_Byts', 'Fwd_Seg_Size_Min']

Near-constant columns (top value > 99%):


,top_value_ratio
Fwd_PSH_Flags,1.000000
Fwd_URG_Flags,1.000000
Bwd_Blk_Rate_Avg,1.000000
Bwd_Pkts/b_Avg,1.000000
Bwd_Byts/b_Avg,1.000000
Fwd_Blk_Rate_Avg,1.000000
Fwd_Pkts/b_Avg,1.000000
Fwd_Byts/b_Avg,1.000000
Init_Fwd_Win_Byts,1.000000
Fwd_Seg_Size_Min,1.000000



Zero ratio in numeric columns:


,zero_ratio
Bwd_Byts/b_Avg,1.000000
Bwd_Blk_Rate_Avg,1.000000
Bwd_Pkts/b_Avg,1.000000
Fwd_Byts/b_Avg,1.000000
Fwd_Pkts/b_Avg,1.000000
...,...
Flow_IAT_Mean,0.000588
Tot_Bwd_Pkts,0.000000
Flow_Pkts/s,0.000000
Init_Fwd_Win_Byts,0.000000


## Discovery Findings

From the initial inspection, the IoTID20 dataset contains **625,783 rows** and **86 columns**. The schema includes a mix of numeric traffic statistics and raw identifier/string fields such as `Flow_ID`, `Src_IP`, `Dst_IP`, and `Timestamp`. The label structure supports both binary and multiclass settings through `Label`, `Cat`, and `Sub_Cat`.

Several important data quality and preprocessing issues were identified:

- There are **164,087 duplicated rows**, so duplicate removal should be part of the preprocessing pipeline.
- No standard missing values were detected using `isna()`, but there are **infinite values** in at least:
  - `Flow_Byts/s`
  - `Flow_Pkts/s`
  These need to be converted or handled explicitly before scaling or model training.
- The dataset is **strongly imbalanced** at the binary level:
  - `Anomaly`: 585,710 samples (`93.6%`)
  - `Normal`: 40,073 samples (`6.4%`)
- The multiclass labels are also imbalanced:
  - `Cat` is dominated by `Mirai`
  - `Sub_Cat` is dominated by a small number of attack subtypes such as `Mirai-UDP Flooding` and `Mirai-Hostbruteforceg`

Feature inspection also shows that several columns are not useful in their current form:

- The following columns are **constant** and can be dropped:
  - `Fwd_PSH_Flags`
  - `Fwd_URG_Flags`
  - `Fwd_Byts/b_Avg`
  - `Fwd_Pkts/b_Avg`
  - `Fwd_Blk_Rate_Avg`
  - `Bwd_Byts/b_Avg`
  - `Bwd_Pkts/b_Avg`
  - `Bwd_Blk_Rate_Avg`
  - `Init_Fwd_Win_Byts`
  - `Fwd_Seg_Size_Min`
- Some additional columns are **near-constant**, especially several flag-related features such as:
  - `URG_Flag_Cnt`
  - `Bwd_URG_Flags`
  - `ECE_Flag_Cnt`
  - `CWE_Flag_Count`
  - `RST_Flag_Cnt`
  - `FIN_Flag_Cnt`
- Many numeric features are **zero-heavy**, suggesting sparse or highly skewed traffic statistics.

The numeric summaries also indicate strong skewness and large outliers in many traffic features, so scaling or normalization will be necessary before model training.

For preprocessing planning, the current decision is:

- Drop raw identifier / leakage-prone columns:
  - `Flow_ID`
  - `Src_IP`
  - `Dst_IP`
  - `Src_Port`
  - `Dst_Port`
  - `Timestamp`
- Remove duplicated rows
- Replace `inf` / `-inf` values
- Drop constant columns
- Reassess whether near-constant columns should also be removed
- Apply label-specific preprocessing depending on whether the task is binary (`Label`) or multiclass (`Cat`, `Sub_Cat`)
- Apply scaling after cleaning


# Initial Cleaning

In [3]:
df_clean = df.copy()

print("Original shape:", df_clean.shape)

# 1. Drop exact duplicate rows on raw data first
raw_duplicate_rows = df_clean.duplicated().sum()
print("Raw duplicate rows to remove:", raw_duplicate_rows)

df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print("Shape after dropping raw duplicates:", df_clean.shape)

# 2. Drop raw identifier / leakage-prone columns
drop_cols = [
    "Flow_ID",
    "Src_IP",
    "Dst_IP",
    "Src_Port",
    "Dst_Port",
    "Timestamp",
]
drop_cols = [col for col in drop_cols if col in df_clean.columns]
print("\nColumns to drop:", drop_cols)

df_clean = df_clean.drop(columns=drop_cols, errors="ignore")
print("Shape after dropping selected raw columns:", df_clean.shape)

# 3. Replace inf/-inf with NaN, then drop rows containing missing values
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

missing_after_inf = df_clean.isna().sum().sort_values(ascending=False)
print("\nMissing values after replacing inf/-inf with NaN:")
display(missing_after_inf[missing_after_inf > 0].to_frame("missing_count"))

rows_before_nan_drop = len(df_clean)
df_clean = df_clean.dropna().reset_index(drop=True)
rows_after_nan_drop = len(df_clean)

print("\nRows before dropping NaN:", rows_before_nan_drop)
print("Rows after dropping NaN:", rows_after_nan_drop)
print("Removed rows with NaN:", rows_before_nan_drop - rows_after_nan_drop)

# 4. Drop constant and near-constant columns
top_value_ratios = {}
for col in df_clean.columns:
    top_value_ratios[col] = df_clean[col].value_counts(normalize=True, dropna=False).iloc[0]

top_value_ratios = pd.Series(top_value_ratios).sort_values(ascending=False)

constant_like_cols = top_value_ratios[top_value_ratios > 0.999].index.tolist()
constant_like_cols = [col for col in constant_like_cols if col not in ["Label", "Cat", "Sub_Cat"]]

print("\nColumns with top-value ratio > 0.999 to drop:")
print(constant_like_cols)

df_clean = df_clean.drop(columns=constant_like_cols, errors="ignore")
print("Shape after dropping constant / near-constant columns:", df_clean.shape)

# 5. Final report
print("\nRemaining dtypes:")
display(df_clean.dtypes.to_frame("dtype").T)

print("\nRemaining object columns:")
print(df_clean.select_dtypes(include=["object"]).columns.tolist())

print("\nFinal shape after initial cleaning:", df_clean.shape)

print("\nHead of cleaned dataframe:")
display(df_clean.head())


Original shape: (625783, 86)
Raw duplicate rows to remove: 164087
Shape after dropping raw duplicates: (461696, 86)

Columns to drop: ['Flow_ID', 'Src_IP', 'Dst_IP', 'Src_Port', 'Dst_Port', 'Timestamp']
Shape after dropping selected raw columns: (461696, 80)

Missing values after replacing inf/-inf with NaN:


,missing_count
Flow_Pkts/s,323
Flow_Byts/s,323



Rows before dropping NaN: 461696
Rows after dropping NaN: 461373
Removed rows with NaN: 323

Columns with top-value ratio > 0.999 to drop:
['Fwd_Byts/b_Avg', 'Bwd_Byts/b_Avg', 'Fwd_Blk_Rate_Avg', 'Bwd_Blk_Rate_Avg', 'Bwd_Pkts/b_Avg', 'Fwd_Pkts/b_Avg', 'Fwd_Seg_Size_Min', 'Init_Fwd_Win_Byts', 'Fwd_URG_Flags', 'Fwd_PSH_Flags', 'URG_Flag_Cnt', 'Bwd_URG_Flags', 'ECE_Flag_Cnt', 'CWE_Flag_Count', 'RST_Flag_Cnt', 'FIN_Flag_Cnt']
Shape after dropping constant / near-constant columns: (461373, 64)

Remaining dtypes:


,Protocol,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,TotLen_Fwd_Pkts,TotLen_Bwd_Pkts,Fwd_Pkt_Len_Max,Fwd_Pkt_Len_Min,Fwd_Pkt_Len_Mean,Fwd_Pkt_Len_Std,...,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Cat,Sub_Cat
dtype,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,...,float64,float64,float64,float64,float64,float64,float64,object,object,object



Remaining object columns:
['Label', 'Cat', 'Sub_Cat']

Final shape after initial cleaning: (461373, 64)

Head of cleaned dataframe:


,Protocol,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,TotLen_Fwd_Pkts,TotLen_Bwd_Pkts,Fwd_Pkt_Len_Max,Fwd_Pkt_Len_Min,Fwd_Pkt_Len_Mean,Fwd_Pkt_Len_Std,...,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Cat,Sub_Cat
0,17,75,1,1,982.0,1430.0,982.0,982.0,982.0,0.000000,...,0.0,0.0,0.0,75.0,0.000000,75.0,75.0,Anomaly,Mirai,Mirai-Ackflooding
1,6,5310,1,2,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,2655.0,2261.327486,4254.0,1056.0,Anomaly,DoS,DoS-Synflooding
2,6,141,0,3,0.0,2806.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,70.5,0.707107,71.0,70.0,Anomaly,Scan,Scan Port OS
3,6,151,0,2,0.0,2776.0,0.0,0.0,0.0,0.000000,...,0.0,0.0,0.0,151.0,0.000000,151.0,151.0,Anomaly,Mirai,Mirai-Hostbruteforceg
4,17,153,2,1,886.0,420.0,452.0,434.0,443.0,12.727922,...,0.0,0.0,0.0,76.5,0.707107,77.0,76.0,Anomaly,Mirai,Mirai-Hostbruteforceg


# Task Definition

In [4]:
TARGET_COLUMN = "Cat"

label_columns = [col for col in ["Label", "Cat", "Sub_Cat"] if col in df_clean.columns]

y = df_clean[TARGET_COLUMN].copy()
X = df_clean.drop(columns=label_columns, errors="ignore").copy()

print("Selected target:", TARGET_COLUMN)
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget class distribution:")
target_counts = y.value_counts(dropna=False)
target_ratios = (target_counts / len(y)).round(4)
display(pd.DataFrame({"count": target_counts, "ratio": target_ratios}))

print("\nFeature dtypes:")
display(X.dtypes.to_frame("dtype").T)

print("\nRemaining object columns in X:")
object_cols = X.select_dtypes(include=["object"]).columns.tolist()
print(object_cols)


Selected target: Cat
Feature shape: (461373, 61)
Target shape: (461373,)

Target class distribution:


,count,ratio
Cat,,
Mirai,280779,0.6086
DoS,59390,0.1287
Scan,56744,0.1230
Normal,38598,0.0837
MITM ARP Spoofing,25862,0.0561



Feature dtypes:


,Protocol,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,TotLen_Fwd_Pkts,TotLen_Bwd_Pkts,Fwd_Pkt_Len_Max,Fwd_Pkt_Len_Min,Fwd_Pkt_Len_Mean,Fwd_Pkt_Len_Std,...,Init_Bwd_Win_Byts,Fwd_Act_Data_Pkts,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min
dtype,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,...,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64



Remaining object columns in X:
[]


# Data Splitting and Scaling

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE_WITHIN_TRAIN = 0.2

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=VAL_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

print("Split shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

print("\nClass distribution after split:")

for split_name, target in [
    ("train", y_train),
    ("val", y_val),
    ("test", y_test),
]:
    print(f"\n{split_name}:")
    counts = target.value_counts(dropna=False)
    ratios = (counts / len(target)).round(4)
    display(pd.DataFrame({"count": counts, "ratio": ratios}))

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index,
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

print("\nScaled feature shapes:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)


Split shapes:
X_train: (295278, 61) y_train: (295278,)
X_val: (73820, 61) y_val: (73820,)
X_test: (92275, 61) y_test: (92275,)

Class distribution after split:

train:


,count,ratio
Cat,,
Mirai,179698,0.6086
DoS,38010,0.1287
Scan,36316,0.1230
Normal,24702,0.0837
MITM ARP Spoofing,16552,0.0561



val:


,count,ratio
Cat,,
Mirai,44925,0.6086
DoS,9502,0.1287
Scan,9079,0.1230
Normal,6176,0.0837
MITM ARP Spoofing,4138,0.0561



test:


,count,ratio
Cat,,
Mirai,56156,0.6086
DoS,11878,0.1287
Scan,11349,0.1230
Normal,7720,0.0837
MITM ARP Spoofing,5172,0.0560



Scaled feature shapes:
X_train_scaled: (295278, 61)
X_val_scaled: (73820, 61)
X_test_scaled: (92275, 61)


# Label Encoding


In [6]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("Classes:")
print(label_encoder.classes_)

print("\nEncoded target shapes:")
print("y_train_encoded:", y_train_encoded.shape)
print("y_val_encoded:", y_val_encoded.shape)
print("y_test_encoded:", y_test_encoded.shape)

label_mapping = pd.DataFrame({
    "class_name": label_encoder.classes_,
    "encoded_label": range(len(label_encoder.classes_))
})

print("\nLabel mapping:")
display(label_mapping)

print("\nEncoded train target sample:")
print(y_train_encoded[:10])


Classes:
['DoS' 'MITM ARP Spoofing' 'Mirai' 'Normal' 'Scan']

Encoded target shapes:
y_train_encoded: (295278,)
y_val_encoded: (73820,)
y_test_encoded: (92275,)

Label mapping:


,class_name,encoded_label
0,DoS,0
1,MITM ARP Spoofing,1
2,Mirai,2
3,Normal,3
4,Scan,4



Encoded train target sample:
[3 4 2 2 4 2 2 2 2 2]


## Feature Selection

In [7]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

K_BEST = 40

selector = SelectKBest(score_func=mutual_info_classif, k=K_BEST)
X_train_fs = pd.DataFrame(
    selector.fit_transform(X_train_scaled, y_train_encoded),
    index=X_train_scaled.index,
)
X_val_fs = pd.DataFrame(
    selector.transform(X_val_scaled),
    index=X_val_scaled.index,
)
X_test_fs = pd.DataFrame(
    selector.transform(X_test_scaled),
    index=X_test_scaled.index,
)

selected_mask = selector.get_support()
selected_features = X_train_scaled.columns[selected_mask].tolist()

X_train_fs.columns = selected_features
X_val_fs.columns = selected_features
X_test_fs.columns = selected_features

print("Selected feature count:", len(selected_features))
print("Selected features:")
print(selected_features)

Selected feature count: 40
Selected features:
['Protocol', 'Flow_Duration', 'Tot_Bwd_Pkts', 'TotLen_Fwd_Pkts', 'TotLen_Bwd_Pkts', 'Fwd_Pkt_Len_Max', 'Fwd_Pkt_Len_Min', 'Fwd_Pkt_Len_Mean', 'Bwd_Pkt_Len_Max', 'Bwd_Pkt_Len_Min', 'Bwd_Pkt_Len_Mean', 'Flow_Byts/s', 'Flow_Pkts/s', 'Flow_IAT_Mean', 'Flow_IAT_Max', 'Flow_IAT_Min', 'Bwd_IAT_Tot', 'Bwd_IAT_Mean', 'Bwd_IAT_Max', 'Bwd_IAT_Min', 'Fwd_Header_Len', 'Bwd_Header_Len', 'Fwd_Pkts/s', 'Bwd_Pkts/s', 'Pkt_Len_Min', 'Pkt_Len_Max', 'Pkt_Len_Mean', 'SYN_Flag_Cnt', 'ACK_Flag_Cnt', 'Pkt_Size_Avg', 'Fwd_Seg_Size_Avg', 'Bwd_Seg_Size_Avg', 'Subflow_Fwd_Byts', 'Subflow_Bwd_Pkts', 'Subflow_Bwd_Byts', 'Init_Bwd_Win_Byts', 'Fwd_Act_Data_Pkts', 'Idle_Mean', 'Idle_Max', 'Idle_Min']


## Imbalance Handling

In [8]:
from collections import Counter
import numpy as np
import pandas as pd
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

# Work only on train split
train_counts = Counter(y_train_encoded)
print("Original train distribution:")
display(pd.Series(train_counts).sort_index().to_frame("count"))

# Mild target: balance toward the median class count
counts_array = np.array(list(train_counts.values()))
target_count = int(np.median(counts_array))
print("\nTarget count per class:", target_count)

X_res = X_train_fs.copy()
y_res = y_train_encoded.copy()

# 1) Undersample classes above target_count
majority_strategy = {cls: target_count for cls, cnt in train_counts.items() if cnt > target_count}
if majority_strategy:
    rus = RandomUnderSampler(sampling_strategy=majority_strategy, random_state=42)
    X_res, y_res = rus.fit_resample(X_res, y_res)

# 2) Oversample classes below target_count
after_counts = Counter(y_res)
minority_strategy = {cls: target_count for cls, cnt in after_counts.items() if cnt < target_count}
if minority_strategy:
    ros = RandomOverSampler(sampling_strategy=minority_strategy, random_state=42)
    X_res, y_res = ros.fit_resample(X_res, y_res)

X_train_bal = pd.DataFrame(X_res, columns=X_train_fs.columns)
y_train_bal = np.array(y_res)

print("\nBalanced train distribution:")
display(pd.Series(Counter(y_train_bal)).sort_index().to_frame("count"))

print("\nShapes:")
print("X_train_bal:", X_train_bal.shape)
print("y_train_bal:", y_train_bal.shape)
print("X_val_fs:", X_val_fs.shape)
print("X_test_fs:", X_test_fs.shape)


Original train distribution:


,count
0,38010
1,16552
2,179698
3,24702
4,36316



Target count per class: 36316

Balanced train distribution:


,count
0,36316
1,36316
2,36316
3,36316
4,36316



Shapes:
X_train_bal: (181580, 40)
y_train_bal: (181580,)
X_val_fs: (73820, 40)
X_test_fs: (92275, 40)


# Prepared Data Summary

In [9]:
prepared_summary = {
    "target_column": TARGET_COLUMN,
    "num_classes": int(len(label_encoder.classes_)),
    "num_features": int(X_train_scaled.shape[1]),
    "train_shape": tuple(X_train_scaled.shape),
    "val_shape": tuple(X_val_scaled.shape),
    "test_shape": tuple(X_test_scaled.shape),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "val_size_within_train": VAL_SIZE_WITHIN_TRAIN,
    "scaler": "StandardScaler",
}

display(pd.DataFrame([prepared_summary]))


,target_column,num_classes,num_features,train_shape,val_shape,test_shape,random_state,test_size,val_size_within_train,scaler
0,Cat,5,61,"(295278, 61)","(73820, 61)","(92275, 61)",42,0.2,0.2,StandardScaler


# Export Prepared Dataset


In [10]:
# from pathlib import Path
# import json

# output_dir = Path("data/1_processed/IoTID20")
# output_dir.mkdir(parents=True, exist_ok=True)

# X_train_scaled.to_csv(output_dir / "X_train.csv", index=False)
# X_val_scaled.to_csv(output_dir / "X_val.csv", index=False)
# X_test_scaled.to_csv(output_dir / "X_test.csv", index=False)

# pd.DataFrame({"Cat": y_train.values, "Cat_encoded": y_train_encoded}).to_csv(
#     output_dir / "y_train.csv", index=False
# )
# pd.DataFrame({"Cat": y_val.values, "Cat_encoded": y_val_encoded}).to_csv(
#     output_dir / "y_val.csv", index=False
# )
# pd.DataFrame({"Cat": y_test.values, "Cat_encoded": y_test_encoded}).to_csv(
#     output_dir / "y_test.csv", index=False
# )

# metadata = {
#     "dataset": "IoTID20",
#     "target_column": TARGET_COLUMN,
#     "classes": label_encoder.classes_.tolist(),
#     "label_mapping": {
#         class_name: int(idx)
#         for idx, class_name in enumerate(label_encoder.classes_)
#     },
#     "dropped_raw_columns": drop_cols,
#     "dropped_constant_like_columns": constant_like_cols,
#     "scaler": "StandardScaler",
#     "random_state": RANDOM_STATE,
#     "test_size": TEST_SIZE,
#     "val_size_within_train": VAL_SIZE_WITHIN_TRAIN,
#     "train_shape": list(X_train_scaled.shape),
#     "val_shape": list(X_val_scaled.shape),
#     "test_shape": list(X_test_scaled.shape),
# }

# with open(output_dir / "metadata.json", "w", encoding="utf-8") as f:
#     json.dump(metadata, f, indent=2, ensure_ascii=True)

# print("Export completed to:", output_dir)
# print("\nExported files:")
# for path in sorted(output_dir.iterdir()):
#     print("-", path.name)

In [11]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Xtr = torch.tensor(X_train_scaled.values, dtype=torch.float32)
Xva = torch.tensor(X_val_scaled.values, dtype=torch.float32)
Xte = torch.tensor(X_test_scaled.values, dtype=torch.float32)
ytr = torch.tensor(y_train_encoded, dtype=torch.long)
yva = torch.tensor(y_val_encoded, dtype=torch.long)
yte = torch.tensor(y_test_encoded, dtype=torch.long)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=512)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=512)

model = nn.Sequential(
    nn.Linear(X_train_scaled.shape[1], 256), nn.ReLU(), nn.Dropout(0.1),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1),
    nn.Linear(128, len(label_encoder.classes_))
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def eval_loader(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in loader:
            p = model(x.to(device)).argmax(1).cpu()
            preds.append(p)
            trues.append(y)
    p = torch.cat(preds).numpy()
    t = torch.cat(trues).numpy()
    return {
        "acc": accuracy_score(t, p),
        "macro_f1": f1_score(t, p, average="macro"),
        "y_true": t,
        "y_pred": p,
    }

best, patience, bad = 0.0, 5, 0
epochs = 20

for epoch in range(1, epochs + 1):
    model.train()
    train_preds, train_trues = [], []

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()

        train_preds.append(logits.argmax(1).detach().cpu())
        train_trues.append(y.detach().cpu())

    train_pred = torch.cat(train_preds).numpy()
    train_true = torch.cat(train_trues).numpy()
    train_acc = accuracy_score(train_true, train_pred)
    val_metrics = eval_loader(val_loader)

    print(
        f"Epoch {epoch:02d} | "
        f"train_acc={train_acc:.4f} | "
        f"val_acc={val_metrics['acc']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["acc"] > best:
        best, bad = val_metrics["acc"], 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            break

model.load_state_dict(best_state)

val_metrics = eval_loader(val_loader)
test_metrics = eval_loader(test_loader)

print("val_acc:", round(val_metrics["acc"], 4), "val_macro_f1:", round(val_metrics["macro_f1"], 4))
print("test_acc:", round(test_metrics["acc"], 4), "test_macro_f1:", round(test_metrics["macro_f1"], 4))

print("\nClassification report (test):")
print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], target_names=label_encoder.classes_))

print("\nConfusion matrix (test):")
display(pd.DataFrame(
    confusion_matrix(test_metrics["y_true"], test_metrics["y_pred"]),
    index=label_encoder.classes_,
    columns=label_encoder.classes_
))


Epoch 01 | train_acc=0.8099 | val_acc=0.8236 | val_macro_f1=0.6907
Epoch 02 | train_acc=0.8302 | val_acc=0.8329 | val_macro_f1=0.7181
Epoch 03 | train_acc=0.8352 | val_acc=0.8353 | val_macro_f1=0.7306
Epoch 04 | train_acc=0.8366 | val_acc=0.8378 | val_macro_f1=0.7287
Epoch 05 | train_acc=0.8377 | val_acc=0.8385 | val_macro_f1=0.7323
Epoch 06 | train_acc=0.8391 | val_acc=0.8412 | val_macro_f1=0.7389
Epoch 07 | train_acc=0.8391 | val_acc=0.8395 | val_macro_f1=0.7301
Epoch 08 | train_acc=0.8405 | val_acc=0.8395 | val_macro_f1=0.7321
Epoch 09 | train_acc=0.8408 | val_acc=0.8368 | val_macro_f1=0.7222
Epoch 10 | train_acc=0.8413 | val_acc=0.8401 | val_macro_f1=0.7320
Epoch 11 | train_acc=0.8417 | val_acc=0.8423 | val_macro_f1=0.7368
Epoch 12 | train_acc=0.8418 | val_acc=0.8404 | val_macro_f1=0.7331
Epoch 13 | train_acc=0.8422 | val_acc=0.8415 | val_macro_f1=0.7366
Epoch 14 | train_acc=0.8420 | val_acc=0.8427 | val_macro_f1=0.7462
Epoch 15 | train_acc=0.8429 | val_acc=0.8432 | val_macro_f1=0.

,DoS,MITM ARP Spoofing,Mirai,Normal,Scan
DoS,11855,0,19,4,0
MITM ARP Spoofing,0,833,2324,32,1983
Mirai,0,56,48576,48,7476
Normal,0,2,421,6544,753
Scan,0,18,1037,25,10269
